# Network analytics with Neo4j Graph Data Science (GDS)

Once you have a knowledge graph, the next question is usually *which nodes matter the most*. This notebook runs four classic graph algorithms against the Neo4j workspace that Tutorial 1 populated, using **Neo4j GDS** procedures called from Cypher.

| Section | Algorithm | Tier |
|---|---|---|
| 1 | `gds.degree.stream` — most direct connections | **Basic** |
| 2 | `gds.pageRank.stream` — recursive importance | **Basic** |
| 3 | `gds.betweenness.stream` — bridges between clusters | **Advanced** |
| 4 | `gds.louvain.stream` — community detection | **Advanced** |
| 5 | Multi-signal visualization (color = community, size = PageRank) | **Advanced** |
| 6 | Edge-type frequency — quick ontology hygiene check | **Basic** |

If you're new to graph analytics, sections **1, 2, 6** give you a complete picture (who's prominent, who's important, are the relationship types healthy). Sections **3-5** add depth: betweenness is the bridge-finder, Louvain is the clusterer, and the visualization combines the two.

**Setup prerequisite.** GDS-Enterprise is the version Neo4j's plugin manifest pulls by default, but it requires Enterprise classes that DozerDB Community doesn't ship. We use **OpenGDS** (DozerDB's Community-compatible distribution) instead. Install it once with:

```bash
make tutorials-gds
```

(That runs `scripts/setup/install-opengds.sh` to drop the OpenGDS jar into the tutorials Neo4j and restart the container.) Verify with `RETURN gds.version()` in the Neo4j Browser at http://localhost:7474.

**Data prerequisite.** Run Tutorial 1 first to populate the `finder_tutorial` workspace. This notebook reads from that workspace; it does not re-extract.

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://tutorials-neo4j:7687")
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "tutorialspw")
WORKSPACE_ID   = os.environ.get("SEOCHO_NETWORK_WORKSPACE", "finder_tutorial")
GRAPH_NAME     = f"network-{WORKSPACE_ID}"  # GDS in-memory projection name

print(f"Neo4j @ {NEO4J_URI}")
print(f"Reading workspace: {WORKSPACE_ID}")
print(f"GDS projection:    {GRAPH_NAME}")

In [ ]:
from seocho.store.graph import Neo4jGraphStore

graph_store = Neo4jGraphStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

# Sanity-check GDS is loaded.
rows = graph_store.query("RETURN gds.version() AS version")
print(f"GDS version: {rows[0]['version']}")

# Sanity-check the workspace is populated.
counts = graph_store.query(
    "MATCH (n) WHERE n._workspace_id = $w RETURN count(n) AS nodes",
    params={"w": WORKSPACE_ID},
)
if counts[0]["nodes"] == 0:
    raise RuntimeError(
        f"Workspace '{WORKSPACE_ID}' is empty. Run Tutorial 1 first "
        "(01_vector_vs_graph_rag.ipynb) so it populates this workspace."
    )
edge_counts = graph_store.query(
    "MATCH ()-[r]->() WHERE r._workspace_id = $w RETURN count(r) AS rels",
    params={"w": WORKSPACE_ID},
)
print(f"Workspace size: {counts[0]['nodes']} nodes, {edge_counts[0]['rels']} relationships")

## Project the workspace into a GDS in-memory graph

GDS algorithms run against an **in-memory graph projection** that you build with `gds.graph.project`. The projection holds only the labels and relationship types you ask for; it's separate from the persisted graph in Neo4j. Once you're done, drop the projection with `gds.graph.drop` to free memory.

Because we only want this workspace's slice of the graph, we use the *Cypher projection* form (`gds.graph.project.cypher`) — it lets us scope the node and relationship queries with `WHERE n._workspace_id = $workspace_id`.

In [ ]:
# Drop any leftover projection from a previous run
try:
    graph_store.execute_write(
        "CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName",
        params={"name": GRAPH_NAME},
    )
except Exception:
    pass

# Native projection (`'*', '*'` = all labels, all rel types). The
# OpenGDS 2.12.0 jar's `gds.graph.project.cypher` form has a known
# LinkageError on DozerDB 5.26.3, so we use the native call and filter
# the results to our workspace at the output stage instead. Results
# downstream apply `WHERE n._workspace_id = $w`.
result = graph_store.execute_write(
    """
    CALL gds.graph.project($name, '*', '*')
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    """,
    params={"name": GRAPH_NAME},
)
print(f"Projected GDS graph: {result}")

## 1. Degree centrality — who has the most connections?

The simplest impactful-node signal: how many edges touch it. `gds.degree.stream` returns a score per node; we join back on the original node properties for readable output.

In [ ]:
rows = graph_store.query(
    """
    CALL gds.degree.stream($name, { orientation: 'UNDIRECTED' })
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE n._workspace_id = $ws
    RETURN n.name AS name, labels(n)[0] AS label, score
    ORDER BY score DESC LIMIT 10
    """,
    params={"name": GRAPH_NAME, "ws": WORKSPACE_ID},
)
print(f"{'rank':4s}  {'name':40s}  {'label':18s}  {'degree':>6s}")
print("-" * 80)
for i, r in enumerate(rows, 1):
    name = (r["name"] or "?")[:40]
    print(f"{i:>4}  {name:40s}  {(r['label'] or '?'):18s}  {int(r['score']):>6d}")

## 2. PageRank — recursive importance

Degree treats every neighbor equally. PageRank doesn't: a node connected to highly-ranked nodes gets a bigger boost than a node connected to obscure ones. Useful for surfacing entities that matter *because* of who they're connected to.

In [ ]:
rows = graph_store.query(
    """
    CALL gds.pageRank.stream($name, { dampingFactor: 0.85, maxIterations: 20 })
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE n._workspace_id = $ws
    RETURN n.name AS name, labels(n)[0] AS label, score
    ORDER BY score DESC LIMIT 10
    """,
    params={"name": GRAPH_NAME, "ws": WORKSPACE_ID},
)
print(f"{'rank':4s}  {'name':40s}  {'label':18s}  {'pagerank':>9s}")
print("-" * 80)
for i, r in enumerate(rows, 1):
    name = (r["name"] or "?")[:40]
    print(f"{i:>4}  {name:40s}  {(r['label'] or '?'):18s}  {r['score']:>9.4f}")

## Advanced — 3. Betweenness centrality: bridges between communities

Betweenness asks a different question from degree and PageRank: how often does this node lie on the shortest path between *other* pairs of nodes? High-betweenness nodes are the bridges — remove them and the graph fragments. In FinDER, an executive who connects two otherwise-disjoint companies will rank here.

This is the most expensive of the four metrics — `O(V·E)` — so on large graphs you usually run it on a sampled or filtered projection rather than the whole thing.

In [ ]:
rows = graph_store.query(
    """
    CALL gds.betweenness.stream($name)
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE n._workspace_id = $ws
    RETURN n.name AS name, labels(n)[0] AS label, score
    ORDER BY score DESC LIMIT 10
    """,
    params={"name": GRAPH_NAME, "ws": WORKSPACE_ID},
)
print(f"{'rank':4s}  {'name':40s}  {'label':18s}  {'betweenness':>11s}")
print("-" * 80)
for i, r in enumerate(rows, 1):
    name = (r["name"] or "?")[:40]
    print(f"{i:>4}  {name:40s}  {(r['label'] or '?'):18s}  {r['score']:>11.4f}")

## 4. Community detection — Louvain

Louvain finds tightly-connected clusters by maximizing modularity. Each node ends up tagged with a `communityId`. Sizes and sample members give you a sense of what each cluster is about.

In [ ]:
rows = graph_store.query(
    """
    CALL gds.louvain.stream($name)
    YIELD nodeId, communityId
    WITH gds.util.asNode(nodeId) AS n, communityId
    WHERE n._workspace_id = $ws
    WITH communityId, collect(n) AS members
    RETURN communityId, size(members) AS size,
           [m IN members | m.name][..6] AS sample
    ORDER BY size DESC
    """,
    params={"name": GRAPH_NAME, "ws": WORKSPACE_ID},
)
print(f"Detected {len(rows)} communities. Top 5 by size:")
for i, r in enumerate(rows[:5], 1):
    sample = ", ".join((s or "?")[:25] for s in r["sample"] or [])
    print(f"  community {r['communityId']:>3}  size={r['size']:>3}  members: {sample}")

# Save (internal nodeId -> communityId) for the visualization next.
node_communities = {
    row["nodeId"]: row["communityId"]
    for row in graph_store.query(
        """
        CALL gds.louvain.stream($name)
        YIELD nodeId, communityId
        WITH gds.util.asNode(nodeId) AS n, nodeId, communityId
        WHERE n._workspace_id = $ws
        RETURN nodeId, communityId
        """,
        params={"name": GRAPH_NAME, "ws": WORKSPACE_ID},
    )
}

## 5. Visualize — color by Louvain community, size by PageRank

Two signals at once: communities tell you *which slice of the world* a node belongs to (color); PageRank tells you *how prominent* it is in that slice (size). We pull the subgraph back from Neo4j into a small NetworkX `DiGraph` purely for layout and matplotlib drawing — the *analytics* (PageRank scores, community ids) all came from GDS.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# Pull (internal_id, nid, name, label, pageRank) for our workspace only.
viz = graph_store.query(
    """
    CALL gds.pageRank.stream($name, { dampingFactor: 0.85, maxIterations: 20 })
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, nodeId, score AS pr
    WHERE n._workspace_id = $ws
    RETURN nodeId AS internal_id, n.id AS nid, n.name AS name,
           labels(n)[0] AS label, pr
    """,
    params={"name": GRAPH_NAME, "ws": WORKSPACE_ID},
)

edges_rows = graph_store.query(
    "MATCH (a)-[r]->(b) WHERE r._workspace_id = $w "
    "RETURN a.id AS src, b.id AS tgt, type(r) AS type",
    params={"w": WORKSPACE_ID},
)

g = nx.DiGraph()
internal_to_nid = {}
pr_by_nid = {}
for r in viz:
    nid = r["nid"] or str(r["internal_id"])
    internal_to_nid[r["internal_id"]] = nid
    pr_by_nid[nid] = r["pr"]
    g.add_node(nid, label=r["label"] or "?", name=(r["name"] or nid)[:40])
for e in edges_rows:
    if e.get("src") and e.get("tgt"):
        if e["src"] not in g:
            g.add_node(e["src"], label="?", name=e["src"][:40])
        if e["tgt"] not in g:
            g.add_node(e["tgt"], label="?", name=e["tgt"][:40])
        g.add_edge(e["src"], e["tgt"], type=e.get("type", "REL"))

# Map nid -> communityId
comm_for_nid = {internal_to_nid[k]: v for k, v in node_communities.items()
                if k in internal_to_nid}

fig, ax = plt.subplots(1, 1, figsize=(13, 9))
pos = nx.spring_layout(g, seed=42, k=0.6)
palette = plt.cm.tab20.colors
node_colors = [palette[comm_for_nid.get(n, 0) % len(palette)] for n in g.nodes()]
node_sizes  = [200 + 8000 * pr_by_nid.get(n, 0.0) for n in g.nodes()]
nx.draw_networkx_nodes(g, pos, node_color=node_colors, node_size=node_sizes, alpha=0.92, ax=ax)
nx.draw_networkx_edges(g, pos, alpha=0.3, arrows=True, ax=ax)

TOP_LABELS = 12
top_pr = sorted(pr_by_nid.items(), key=lambda kv: kv[1], reverse=True)[:TOP_LABELS]
labelled = {nid: g.nodes[nid]["name"][:22] for nid, _ in top_pr if nid in g}
nx.draw_networkx_labels(g, pos, labels=labelled, font_size=8, font_weight="bold", ax=ax)

communities_n = len(set(comm_for_nid.values()))
ax.set_title(
    f"FinDER graph — color: GDS Louvain community ({communities_n} clusters)  ·  size: GDS PageRank",
    fontsize=11,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Edge importance — which relationship types carry the most signal?

Edges have *types* (`EMPLOYS`, `REPORTED`, `MENTIONS`, …). Which type appears most often is a quick proxy for which relationships drive the structure of the graph. We also flag node pairs connected by 2+ different rel types — usually a sign of overlapping semantics in the ontology.

In [ ]:
rows = graph_store.query(
    "MATCH ()-[r]->() WHERE r._workspace_id = $w "
    "RETURN type(r) AS type, count(r) AS cnt ORDER BY cnt DESC",
    params={"w": WORKSPACE_ID},
)
total = sum(r["cnt"] for r in rows)
print(f"{'rank':4s}  {'rel type':24s}  {'count':>6s}  {'share':>6s}")
print("-" * 60)
for i, r in enumerate(rows, 1):
    print(f"{i:>4}  {r['type']:24s}  {r['cnt']:>6d}  {r['cnt']/total:>5.0%}")

redundant = graph_store.query(
    """
    MATCH (a)-[r]->(b) WHERE r._workspace_id = $w
    WITH a, b, collect(distinct type(r)) AS types
    WHERE size(types) >= 2
    RETURN a.name AS src, b.name AS tgt, types LIMIT 10
    """,
    params={"w": WORKSPACE_ID},
)
print(f"\nNode pairs connected by 2+ different relationship types: {len(redundant)}")
for r in redundant:
    print(f"  {(r['src'] or '?')[:25]} -> {(r['tgt'] or '?')[:25]}  via {r['types']}")

## Cleanup the GDS projection

In [ ]:
graph_store.execute_write(
    "CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName",
    params={"name": GRAPH_NAME},
)
print(f"Dropped GDS projection '{GRAPH_NAME}'.")

## What to take away

- **Degree** is fast and intuitive — your most-mentioned company should be near the top.
- **PageRank** is degree's smarter cousin. It surfaces nodes that matter *because* they're connected to other important nodes.
- **Betweenness** identifies bridges. Removing high-betweenness nodes disconnects the graph; they're often executives or documents that link otherwise-separate sub-corpora.
- **Louvain communities** give you a shortcut to summarization. A single sentence per community often captures most of what the graph is about.
- **Edge-type frequency** is a cheap sanity check on the ontology. If 95% of your edges are `MENTIONS`, your ontology is too coarse — most facts collapsed into one bucket. If 50 different rel types each have <2% share, you over-fragmented.

**Why GDS Cypher procedures, not NetworkX in Python.** GDS runs the algorithm *inside Neo4j*, on the projected in-memory graph. That's the right answer when:

- Your graph is too large to ship over Bolt to a Python process.
- You want to write the result back into the graph (`gds.pageRank.write`) so other Cypher queries can use it.
- Your team thinks in Cypher already.

If your graph fits comfortably in memory and you prefer Python ergonomics, NetworkX gives you the same algorithms with less plumbing. Both are valid; pick the one that matches your team's center of gravity.